#### RAG with PRO Techniques

- RAG with Advanced Techinques

    1. No LangChain! Just native for maximum flexibility

    2. Use LLM to divide the chunks in a sensible way

    3. Use LLM to re-write chunks in a way that's most useful ("document pre-processing)



In [1]:
# imports

import os
from dotenv import load_dotenv
from pathlib import Path
from litellm import completion
from pydantic import BaseModel, Field                       # pydantic way of structured outputs
from chromadb import PersistentClient                       # ChromaDB client library
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
import numpy as np
from sklearn.manifold import TSNE                           # projecting chunks to 2D or 3D
import plotly.graph_objects as go

In [22]:
load_dotenv(override=True)

ollama_base_url = 'http://localhost:11434/'
ollama_api_key = os.getenv('OLLAMA_API_KEY')

MODEL = 'ollama/llama3.2'
DB_NAME = '../preprocessed_db'
collection_name = 'docs'
embedding_model = 'all-MiniLM-L6-v2'
KNOWLEDGE_BASE_PATH = Path('../small-knowledge-base')
AVERAGE_CHUNK_SIZE = 500


In [3]:
# For RAG friendly result just like LangChain

class Result(BaseModel):
    page_content : str
    metadata : dict

In [4]:
# class to perfectly represent a chunk

class Chunk(BaseModel):
    headline : str = Field(description="A brief heading for this chunk, typically a few words, that is most likely to be surfaced in a query")
    summary : str = Field(description="A few sentences summarizing the content of this chunk to answer common questions")
    orginal_text : str = Field(description="The original text of this chunk from the provided document, exactly as is, not changed in any way")

    def as_result(self, document):
        metadata = {'source': document['source'], 'type':document['type']}
        return Result(
            page_content=self.headline + '\n\n' + self.summary + '\n\n' + self.orginal_text, metadata=metadata
        )

In [5]:
# class to represent the entire data as a list of Chunk

class Chunks(BaseModel):
    chunks : list[Chunk]

#### Three Steps:

1. Fetch documents from the knowledge base, like LangChain did

2. Call an LLM to turn documents into Chunks

3. Store the Chunks in Chroma

In [6]:
# Step 1
# to fetch the md files from knowledge base into list of dictionaries

def fetch_documents():
    """A homemade version of the LangChain Directory loader"""
    documents = []

    for folder in KNOWLEDGE_BASE_PATH.iterdir():
        doc_type = folder.name
        for file in folder.rglob('*.md'):
            with open(file, 'r', encoding='utf-8') as f:
                documents.append({'type':doc_type, 'source':file.as_posix(), 'text':f.read()})

    print(f'Loaded {len(documents)} documents')
    return documents

''' 
`file.as_posix()`   - returns individual file path using /

'''

' \n`file.as_posix()`   - returns individual file path using /\n\n'

In [7]:
documents = fetch_documents()
documents[0]

Loaded 27 documents


{'type': 'products',
 'source': '../small-knowledge-base/products/Rellm.md',
 'text': "# Product Summary\n\n# Rellm: AI-Powered Enterprise Reinsurance Solution\n\n## Summary\n\nRellm is an innovative enterprise reinsurance product developed by Insurellm, designed to transform the way reinsurance companies operate. Harnessing the power of artificial intelligence, Rellm offers an advanced platform that redefines risk management, enhances decision-making processes, and optimizes operational efficiencies within the reinsurance industry. With seamless integrations and robust analytics, Rellm enables insurers to proactively manage their portfolios and respond to market dynamics with agility.\n\n## Features\n\n### AI-Driven Analytics\nRellm utilizes cutting-edge AI algorithms to provide predictive insights into risk exposures, enabling users to forecast trends and make informed decisions. Its real-time data analysis empowers reinsurance professionals with actionable intelligence.\n\n### Seaml

In [8]:
# Step 2 - use llm to convert the data into efficient chunks for retrieval
# to make a prompt

def make_prompt(document):
    ''' Creates a prompt to ask the llm to convert the data into efficient chunks for retrieval '''
    how_many = (len(document['text']) // AVERAGE_CHUNK_SIZE) + 1
    return f""" 
You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the shared drive of a company called Insurellm.
The document is of type: {document["type"]}
The document has been retrieved from: {document["source"]}

A chatbot will use these chunks to answer questions about the company.
You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
This document should probably be split into {how_many} chunks, but you can have more or less as appropriate.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of the chunk.
Together your chunks should represent the entire document with overlap.

Here is the document:

{document["text"]}

Respond with the chunks.
"""

In [9]:
print(make_prompt(documents[0]))

 
You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the shared drive of a company called Insurellm.
The document is of type: products
The document has been retrieved from: ../small-knowledge-base/products/Rellm.md

A chatbot will use these chunks to answer questions about the company.
You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
This document should probably be split into 8 chunks, but you can have more or less as appropriate.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of the chunk.
Together your chunks should represent the entire document with overlap.

Here is the document:

# Product Summary

# Rellm: AI-Powered E

In [10]:
# to convert the prompt into user_prompt

def make_message(document):
    return [
        {'role':'user', 'content':make_prompt(document)}
    ]

In [11]:
make_message(documents[0])

[{'role': 'user',
  'content': " \nYou take a document and you split the document into overlapping chunks for a KnowledgeBase.\n\nThe document is from the shared drive of a company called Insurellm.\nThe document is of type: products\nThe document has been retrieved from: ../small-knowledge-base/products/Rellm.md\n\nA chatbot will use these chunks to answer questions about the company.\nYou should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.\nThis document should probably be split into 8 chunks, but you can have more or less as appropriate.\nThere should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.\n\nFor each chunk, you should provide a headline, a summary, and the original text of the chunk.\nTogether your chunks should represent the entire document with overlap.\n\nHere is the docu

In [12]:
# using llm to process the document into chunks

def process_documents(document):
    messages = make_message(document)
    response = completion(model=MODEL, messages=messages, response_format=Chunks, base_url=ollama_base_url, api_key=ollama_api_key)
    reply = response.choices[0].message.content
    doc_as_chunks = Chunks.model_validate_json(reply).chunks
    return [chunk.as_result(document) for chunk in doc_as_chunks]

''' 
The LLM generates the response in JSON format.

Then, the JSON output is validated against the `Chunks` Pydantic model and converted into a `Chunks` object. From that object, only the list of `Chunk` objects is extracted.

Then, each `Chunk` in that list is converted into a `Result` object using `chunk.as_result(document)`.

Finally, the entire list of `Result` objects is returned.

'''

' \nThe LLM generates the response in JSON format.\n\nThen, the JSON output is validated against the `Chunks` Pydantic model and converted into a `Chunks` object. From that object, only the list of `Chunk` objects is extracted.\n\nThen, each `Chunk` in that list is converted into a `Result` object using `chunk.as_result(document)`.\n\nFinally, the entire list of `Result` objects is returned.\n\n'

- `response_format=Chunks`            - returns the response as Chunks object in JSON format


      {
        "chunks": [
          {
            "headline": "Visa Requirements",
            "summary": "This section explains the required documents.",
            "original_text": "Applicants must submit a valid passport."
          },
          {
            "headline": "Application Process",
            "summary": "This section explains how to apply.",
            "original_text": "Applications can be submitted online."
          }
        ]
      }



- `Chunks.model_validate_json(reply)`   - validates the json format and converts it to Chunks object i.e., list[Chunk]

      Chunks(
          chunks=[
              Chunk(
                  headline="Visa Requirements",
                  summary="This section explains the required documents.",
                  original_text="Applicants must submit a valid passport."
              ),
              Chunk(
                  headline="Application Process",
                  summary="This section explains how to apply.",
                  original_text="Applications can be submitted online."
              )
          ]
      )


- `Chunks.model_validate_json(reply).chunks`  - extracts only the list[Chunk] from the previous output

      [
          Chunk(
              headline="Visa Requirements",
              summary="This section explains the required documents.",
              original_text="Applicants must submit a valid passport."
          ),
          Chunk(
              headline="Application Process",
              summary="This section explains how to apply.",
              original_text="Applications can be submitted online."
          )
      ]


- `[chunk.as_result(document) for chunk in doc_as_chunks]`    - converts each Chunk object into Result object and returns the output as list of Result objects

      [
          Result(
              page_content="Headline + Summary + Original Text",
              metadata={"source": "...", "type": "..."}
          ),
          Result(
              page_content="Headline + Summary + Original Text",
              metadata={"source": "...", "type": "..."}
          )
      ]

In [13]:
process_documents(documents[0])

[Result(page_content='Product Overview\n\nRellm is an innovative enterprise reinsurance product developed by Insurellm, designed to transform the way reinsurance companies operate.\n\n# Product Summary\\n\\n# Rellm: AI-Powered Enterprise Reinsurance Solution\n\\n## Summary\\nRellm is an innovative enterprise reinsurance product developed by Insurellm, designed to transform the way reinsurance companies operate. Harnessing the power of artificial intelligence, Rellm offers an advanced platform that redefines risk management, enhances decision-making processes, and optimizes operational efficiencies within the reinsurance industry. With seamless integrations and robust analytics, Rellm enables insurers to proactively manage their portfolios and respond to market dynamics with agility.', metadata={'source': '../small-knowledge-base/products/Rellm.md', 'type': 'products'}),
 Result(page_content="Key Features\n\nRellm offers a range of features that enable reinsurance professionals to make 

In [14]:
# to create the chunks for entire documents data

from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm


def create_chunks(documents, max_workers=5):

    chunks = []

    with ThreadPoolExecutor(max_workers=max_workers) as executor:       # multi threading

        results = executor.map(process_documents, documents)

        for result in tqdm(results, total=len(documents)):              # tqdm gives the progress bar showing how far the loop is progressed
            chunks.extend(result)

    return chunks

In [15]:
chunks = create_chunks(documents, max_workers=5)

100%|██████████| 27/27 [32:19<00:00, 71.83s/it]  


In [16]:
# Above code can also be written without concurrent threads incase of ratelimit errors from the model provider

''' 
def create_chunks(documents):
    chunks = []

    for doc in tqdm(documents):
        chunks.extend(process_documents(doc))
    return chunks

'''

' \ndef create_chunks(documents):\n    chunks = []\n\n    for doc in tqdm(documents):\n        chunks.extend(process_documents(doc))\n    return chunks\n\n'

In [23]:
# Step 3 - Creating Vector DB

def create_embeddings(chunks):
    chroma = PersistentClient(path=DB_NAME)
    if collection_name in [c.name for c in chroma.list_collections()]:
        chroma.delete_collection(collection_name)

    texts = [chunk.page_content for chunk in chunks]
    embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    vectors = embedding_model.encode(texts).tolist()

    collection = chroma.get_or_create_collection(collection_name)

    ids = [str(i) for i in range(len(chunks))]
    metas = [chunk.metadata for chunk in chunks]

    collection.add(ids=ids, embeddings=vectors, documents=texts, metadatas=metas)
    print(f'VectorStore created with {collection.count()} documents')
    

In [24]:
create_embeddings(chunks)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

VectorStore created with 231 documents


#### To project the embeddings from higher dimension to 2D or 3D using tSNE

In [25]:
# Setting up the data for projection

chroma = PersistentClient(path=DB_NAME)
collection = chroma.get_or_create_collection(collection_name)
result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['type'] for metadata in metadatas]
colors = [['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)] for t in doc_types]

In [26]:
# 2D using tSNE

tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# creating 2D
fig = go.Figure(data=[go.Scatter(
    x = reduced_vectors[:,0],
    y = reduced_vectors[:,1],
    mode = 'markers',
    marker = dict(size=5, color=colors, opacity = 0.8),
    text = [f'Type : {t}<br>Text : {d[:100]}...' for t,d in zip(doc_types, documents)],
    hoverinfo = 'text'
)])

fig.update_layout(title='2D Chroma Vector DB Visualization',
    scene = dict(xaxis_title='x', yaxis_title='y'),
    width = 800,
    height = 600,
    margin = dict(r=20, l=10, b=10, t=40)
)

fig.show()


In [27]:
# 3D using tSNE

tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()

#### Building the Advanced RAG

We will use the following techniques:

1. Re-ranking - reorder the rank results

2. Query re-writing

In [ ]:
# RankOrder object using Pydantic model

class RankOrder(BaseModel):
    order: list[int] = Field(
        description = 'The order of relevance of chunks, from most relevant to least relevant, by chunk id number'
        )

In [ ]:
# Re-ranking the response chunks based on the question using LLM

def re_rank(question, chunks):
    system_prompt = """
You are a document re-ranker.
You are provided with a question and a list of relevant chunks of text from a query of a knowledge base.
The chunks are provided in the order they were retrieved; this should be approximately ordered by relevance, but you may be able to improve on that.
You must rank order the provided chunks by relevance to the question, with the most relevant chunk first.
Reply only with the list of ranked chunk ids, nothing else. Include all the chunk ids you are provided with, reranked.
"""
    user_prompt = f"The user has asked the following question:\n\n{question}\n\nOrder all the chunks of text by relevance to the question, from most relevant to least relevant. Include all the chunk ids you are provided with, reranked.\n\n"
    user_prompt += "Here are the chunks:\n\n"
    for index, chunk in enumerate(chunks):
        user_prompt += f"# Chunk ID :{index + 1} : \n\n {chunks.page_content}\n\n"
    user_prompt =+ "Reply only with the list of ranked chunk ids, nothing else."
    messages = [
        {'role':'system', 'content':system_prompt},
        {'role':'user', 'content':user_prompt}
    ]
    response = completion(model=MODEL, messages=messages, base_url=ollama_base_url, api_key=ollama_api_key, response_format=RankOrder)
    reply = response.choices[0].message.content
    order = RankOrder.model_validate_json(reply).order
    print(order)
    return [chunk[i-1] for i in order]